# Notebook Colab mẫu — DataDream, bước 3 (train CLIP)

Dựng môi trường bước 3 (`classify/`) trên Colab từ đầu: clone repo, cài thư viện, nạp
Secrets, tải ảnh sinh từ Kaggle, tải ảnh thật, ghi `classify/local.yaml`, rồi chạy **bài
kiểm nhanh** (ô 10: kiểm tập test + CLIP zero-shot EuroSAT, vài phút) để chắc mọi thứ chạy
được trước khi train thật.

**Cách dùng**

1. **Đừng sửa file này.** Mở nó trên Colab
   (<https://colab.research.google.com/github/Vietpn1909/DataDream/blob/master/notebooks/template_colab.ipynb>),
   rồi *File > Save a copy in GitHub* thành `notebooks/<tên>_<việc>.ipynb` **trên nhánh của
   bạn** (vd `notebooks/tv4_clip.ipynb` trên nhánh `tv4`). Không sửa notebook của người khác
   (CONTRIBUTING.md §6).
2. *Runtime > Change runtime type* → **T4 GPU** (notebook đã đặt sẵn, kiểm lại cho chắc).
3. Secrets (biểu tượng chìa khoá ở thanh bên trái), bật *Notebook access* cho notebook này:
   `KAGGLE_USERNAME`, `KAGGLE_KEY` (kaggle.com → Settings → API → Create New Token), tuỳ
   chọn `WANDB_API_KEY`.
4. Sửa ô **Tham số** (ô 2), rồi *Runtime > Run all*.

**Quy tắc**

- **Không dán API key hay token vào bất kỳ ô nào** — chỉ dùng Secrets. Notebook này không
  bao giờ in giá trị của chúng.
- Trước khi *File > Save a copy in GitHub*: *Edit > Clear all outputs* — output ảnh làm repo
  phình to và diff vô dụng.
- Lưu vào **nhánh của bạn**, không lưu vào `master`.
- Ảnh thật tải vào `/content/data` (ổ đĩa của máy Colab), không đọc thẳng từ Drive: đọc hàng
  nghìn file nhỏ qua Drive rất chậm. Drive chỉ dùng để giữ kết quả (ô 12).

In [ ]:
# ===== 2. Tham so - sua o day =====
BRANCH = "master"   # nhanh se checkout, vd "tv4" de chay code tren nhanh cua ban
WORK = "/content"   # o dia cua may Colab: nhanh, mat khi het phien
DRIVE_OUT = "/content/drive/MyDrive/DataDream/outputs"  # log, checkpoint, CSV chep ve day (ben lau)
TEAM_DRIVE = "/content/drive/MyDrive/DataDream"         # ban du phong (vd split_zhou_EuroSAT.json)

FETCH_KAGGLE = True         # anh sinh + tap few-shot da ban giao tren Kaggle (~4,3 GB)
FETCH_EUROSAT = True        # EuroSAT (~90 MB) + file chia CoOp - can cho bai kiem o o 10
FETCH_AIRCRAFT = True       # FGVC-Aircraft (~2,7 GB) - tap test Aircraft-20
CACHE_AIRCRAFT_TAR = False  # True: giu file tar.gz cua Aircraft tren Drive, lan sau chi giai nen

DATA = f"{WORK}/data"       # anh that
SYN = f"{WORK}/synthetic"   # anh sinh (symlink vao ban Kaggle) = synth_train_data_dir
OUT = f"{WORK}/outputs"     # output cua classify, chep ve DRIVE_OUT o o 12

In [ ]:
# ===== 3. O dau tien cua moi notebook (CONTRIBUTING.md muc 6, giu nguyen) =====
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/DataDream'
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/Vietpn1909/DataDream.git {REPO}
%cd {REPO}

# them: chuyen sang nhanh BRANCH va in commit dang chay (ghi vao cot `commit` cua results/)
!git checkout -q {BRANCH} && git pull -q
COMMIT = !git rev-parse --short=7 HEAD
print("nhanh", BRANCH, "| commit", COMMIT[0])

In [ ]:
# ===== 4. Kiem GPU =====
import sys
import torch
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true
print("python", sys.version.split()[0], "| torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CANH BAO: KHONG CO GPU. Runtime > Change runtime type > T4 GPU, roi chay lai tu dau.")
    print("(Bai kiem o o 10 van chay duoc tren CPU nhung cham; train CLIP thi phai co GPU.)")

In [ ]:
# ===== 5. Thu vien buoc 3 - dung nhu docs/setup.md "Buoc 3: Colab" =====
# KHONG cai lai torch/torchvision: Colab co san, cai lai se pha moi truong (~10 phut).
# requirements-clf.txt ghim numpy==1.26.4, co the HA numpy cua Colab. Neu pip doi numpy
# (dong in ra ben duoi), lam: Runtime > Restart session, chay lai o 2 va o 3 (Tham so, clone),
# roi chay tiep tu o 6. Neu phai doi mot phien ban da ghim: sua thang requirements-clf.txt
# trong mot PR [clf] (header cua file do yeu cau vay), dung chi sua trong notebook.
import importlib.metadata as md
numpy_before = md.version("numpy")
!pip install -q -r requirements-clf.txt
!pip install -q git+https://github.com/openai/CLIP.git
numpy_after = md.version("numpy")
if numpy_after != numpy_before:
    print("pip da doi numpy %s -> %s: Runtime > Restart session, chay lai o 2 va o 3, roi chay tiep tu o 6."
          % (numpy_before, numpy_after))
else:
    print("numpy", numpy_after, "- khong can restart")

In [ ]:
# ===== 6. Secrets -> bien moi truong (KHONG BAO GIO in gia tri) =====
import os
from google.colab import userdata
for name, required in (("KAGGLE_USERNAME", True), ("KAGGLE_KEY", True), ("WANDB_API_KEY", False)):
    try:
        os.environ[name] = userdata.get(name)
        print("%-16s da nap" % name)
    except Exception:  # chua tao secret, hoac chua bat Notebook access
        print("%-16s %s" % (name, "THIEU: them o Secrets (bieu tuong chia khoa), bat Notebook access"
                            if required else "khong co (khong bat buoc)"))

In [ ]:
# ===== 7. Anh sinh (va tap few-shot) da ban giao tren Kaggle =====
# NGUON: docs/data.md muc "Dung tren Colab". Bang map duoi day phai GIONG HET ban o do;
# doi cach dat ten thi sua docs/data.md truoc, roi chep lai vao day.
KAGGLE_ROOT = None
if FETCH_KAGGLE:
    try:
        import kagglehub
    except ImportError:
        !pip install -q kagglehub
        import kagglehub
    import os
    root = kagglehub.dataset_download("vitphmnguyn/datadream") + "/handoff"
    for src, dst in {
        "eurosat_sdbase_K0_n200":    "eurosat/sd2.1/gs2.0_nis50/shot0_template1",
        "eurosat_dset_K16_n200":     "eurosat/sd2.1/gs2.0_nis50/shot16_seed0_template1_lr0.0001_ep200_dswise",
        "eurosat_cls_K16_n200":      "eurosat/sd2.1/gs2.0_nis50/shot16_seed0_template1_lr0.0001_ep200",
        "aircraft20_sdbase_K0_n200": "fgvc_aircraft/sd2.1/gs2.0_nis50/shot0_template1",
    }.items():
        os.makedirs(os.path.dirname(f"{SYN}/{dst}"), exist_ok=True)
        if not os.path.exists(f"{SYN}/{dst}"):
            os.symlink(f"{root}/synthetic/{src}", f"{SYN}/{dst}")
    KAGGLE_ROOT = root
    print("Kaggle:", root)
    !ls {root}/synthetic {root}/real

In [ ]:
# ===== 8. Anh that -> /content/data (o dia may Colab, KHONG phai Drive) =====
import json, os, shutil
import torchvision as tv
os.makedirs(DATA, exist_ok=True)

if FETCH_EUROSAT:
    eu = f"{DATA}/eurosat/train"
    tv.datasets.EuroSAT(root=eu, download=True)  # -> eu/eurosat/2750/<lop>/*.jpg
    split = f"{eu}/split_zhou_EuroSAT.json"       # file chia cua CoOp: classify cham diem tren phan 'test'

    def split_ok(p):
        try:
            with open(p) as f:
                return len(json.load(f)["test"]) == 8100
        except Exception:
            return False

    if not split_ok(split):
        import gdown  # link trong DATASETS.md cua CoOp
        try:
            gdown.download(id="1Ip7yaCWFi0eaOFUGga0lUdVi_DDQth1o", output=split, quiet=True)
        except Exception as e:
            print("gdown loi:", e)
    if not split_ok(split) and os.path.exists(f"{TEAM_DRIVE}/split_zhou_EuroSAT.json"):
        shutil.copy(f"{TEAM_DRIVE}/split_zhou_EuroSAT.json", split)  # ban du phong tren Drive nhom
    print("EuroSAT:", "ok" if split_ok(split) else "THIEU split_zhou_EuroSAT.json -> chep vao " + split)

if FETCH_AIRCRAFT:
    ac = f"{DATA}/fgvc_aircraft/train"
    tar = "fgvc-aircraft-2013b.tar.gz"
    cache = f"{TEAM_DRIVE}/cache/{tar}"
    os.makedirs(ac, exist_ok=True)
    if CACHE_AIRCRAFT_TAR and os.path.exists(cache) and not os.path.isdir(f"{ac}/fgvc-aircraft-2013b"):
        shutil.copy(cache, f"{ac}/{tar}")  # torchvision thay file tar co san -> chi giai nen, khong tai
    # dung loi goi va thu muc ma classify/data.py dung
    tv.datasets.FGVCAircraft(root=ac, split="trainval", annotation_level="variant", download=True)
    if CACHE_AIRCRAFT_TAR and not os.path.exists(cache) and os.path.exists(f"{ac}/{tar}"):
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        shutil.copy(f"{ac}/{tar}", cache)
    print("FGVC-Aircraft: ok")

# Tap few-shot: symlink vao ban Kaggle, de cau truc giong may Viet (<data>/<ds>/real_train_fewshot/seed0)
for ds, name in (("eurosat", "eurosat_fewshot_seed0_K16"), ("fgvc_aircraft", "aircraft20_fewshot_seed0_K16")):
    link = f"{DATA}/{ds}/real_train_fewshot"
    src = f"{KAGGLE_ROOT}/real/{name}" if KAGGLE_ROOT else None
    if src and os.path.isdir(f"{src}/seed0") and not os.path.exists(link):
        os.makedirs(os.path.dirname(link), exist_ok=True)
        os.symlink(src, link)
    print("few-shot %-14s %s" % (ds, link if os.path.isdir(f"{link}/seed0")
                                 else "CHUA CO (%s chua co tren Kaggle, hoac FETCH_KAGGLE=False)" % name))

In [ ]:
# ===== 9. classify/local.yaml (nam trong .gitignore, khong bao gio commit) =====
# classify/config.py doc CA 7 key (configs/local.example.yaml khoi [3]); thieu mot key la KeyError.
import os, yaml
local = {
    # main.py --log=wandb dung key nay. Lay tu Secrets, chi ghi vao file tren may Colab, khong in ra.
    "wandb_key": os.environ.get("WANDB_API_KEY") or "UNUSED_IF_LOG_IS_TENSORBOARD",
    "metadata_dir": "metadata/imagenet",
    "clip_download_dir": f"{WORK}/models/clip",
    "synth_train_data_dir": SYN,
    "real_train_data_dir": {"eurosat": f"{DATA}/eurosat", "fgvc_aircraft": f"{DATA}/fgvc_aircraft"},
    "real_test_data_dir": {"eurosat": f"{DATA}/eurosat", "fgvc_aircraft": f"{DATA}/fgvc_aircraft"},
    # code tu noi them "<fewshot_seed>/" (vd seed0) vao sau
    "real_train_fewshot_data_dir": {"eurosat": f"{DATA}/eurosat/real_train_fewshot",
                                    "fgvc_aircraft": f"{DATA}/fgvc_aircraft/real_train_fewshot"},
}
with open(f"{REPO}/classify/local.yaml", "w") as f:
    yaml.safe_dump(local, f, sort_keys=False)
print(yaml.safe_dump({k: v for k, v in local.items() if k != "wandb_key"}, sort_keys=False))

In [ ]:
# ===== 10. Bai kiem nhanh (vai phut, khong can anh sinh) =====
# (1) tap test / tap that ma classify nap dung chua; (2) CLIP zero-shot EuroSAT tren dung tap
# test do. Ky vong: "KET QUA: DAT", va EuroSAT 8100 anh, top-1 khoang 40% (prompt classify) /
# 48% (prompt reference). Khong truyen --csv o day: chi de kiem moi truong.
%cd {REPO}
!python scripts/test_eval_splits.py --data {DATA}
!python scripts/zeroshot_clip.py --dataset eurosat

In [ ]:
# ===== 11. Vi du chay classify (dang comment) =====
# TV4 phu trach hyperparameter that (--lr, --wd, --epochs, --batch_size...): lay bo so TV4 da
# chot trong notebook/PR cua TV4, DUNG chay voi gia tri mac dinh, dung tu che. Moi co duoi day
# deu co trong classify/config.py. PHAI cd classify: main.py mo local.yaml bang duong dan tuong doi.
# Vi du duoi train tren bo SD goc Aircraft-20 (shot0_template1, 200 anh/lop).
# Chay dai tren Colab free de bi ngat: co the dat --output_dir={DRIVE_OUT} de ghi thang len Drive.
#
# %cd {REPO}/classify
# !python main.py --model_type=clip --dataset=fgvc_aircraft --is_synth_train=True \
#     --sd_version=sd2.1 --n_shot=0 --n_img_per_cls=200 --log=tensorboard --output_dir={OUT}
# %cd {REPO}

In [ ]:
# ===== 12. Luu ket qua =====
# (a) chep log + checkpoint (OUT) ve Drive - /content mat khi het phien
!mkdir -p "{DRIVE_OUT}" && cp -r {OUT}/. "{DRIVE_OUT}/" 2>/dev/null; ls "{DRIVE_OUT}"

# (b) ghi mot dong vao file CSV CUA BAN (results/README.md: moi nguoi mot file, append cuoi
# file), dung header results/results_template.csv; date va commit dien tu dong.
import csv, datetime, os, subprocess
RESULTS_CSV = None  # vd f"{REPO}/results/clf_main.csv" (TV4) - xem results/README.md
row = {"run_name": "aircraft20_sdbase_K0_n200", "dataset": "fgvc_aircraft", "variant": "sdbase",
       "K": 0, "n_img": 200, "seed": "", "setting": "clip-lora", "acc": None, "note": ""}  # acc: top-1 % tren tap test
if RESULTS_CSV and row["acc"] is not None:
    with open(f"{REPO}/results/results_template.csv", encoding="utf-8-sig") as f:
        header = next(csv.reader(f))
    row["date"] = datetime.date.today().isoformat()
    row["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "--short=7", "HEAD"],
                                   capture_output=True, text=True).stdout.strip()
    new = not os.path.exists(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=header, lineterminator="\n")
        if new:
            w.writeheader()
        w.writerow(row)
    !cp "{RESULTS_CSV}" "{DRIVE_OUT}/"
    print("Da ghi", RESULTS_CSV, "va chep ve Drive. Commit file CSV nay vao nhanh cua ban (tu may minh).")
else:
    print("Dien RESULTS_CSV va row (acc, note...) roi chay lai o nay.")
print("Truoc khi luu notebook: Edit > Clear all outputs, roi File > Save a copy in GitHub vao NHANH CUA BAN.")